# Fase 1 — Almacenamiento distribuido
## Ingesta y particionado de datos

**Pipeline:** CC-News + MIND Large → Parquet particionado (simulación HDFS local)

**Fuentes:**
- CC-News: ~3.5 GB, sin etiquetas → usado en Fase 3 (LSH)
- MIND Large: ~1.5 GB comprimido, 18 categorías etiquetadas → usado en Fase 4 (MLP)

## 0. Configuración de rutas

Todo el proyecto vive en local. Solo necesitas tener ~10 GB libres en disco.

In [ ]:
import os

# ── RAÍZ DEL PROYECTO ─────────────────────────────────────────────────────────
PROJECT_PATH = "/Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos"
# ─────────────────────────────────────────────────────────────────────────────

# Datos crudos
RAW_CC         = os.path.join(PROJECT_PATH, "datos", "raw", "cc_news")
MIND_TRAIN_TSV = os.path.join(PROJECT_PATH, "datos", "MINDlarge_train", "news.tsv")
MIND_DEV_TSV   = os.path.join(PROJECT_PATH, "datos", "MINDlarge_dev",   "news.tsv")

# Datos procesados (Parquet)
PROC_CC   = os.path.join(PROJECT_PATH, "datos", "processed", "cc_news")
PROC_MIND = os.path.join(PROJECT_PATH, "datos", "processed", "mind_large")

for path in [RAW_CC, PROC_CC, PROC_MIND]:
    os.makedirs(path, exist_ok=True)

print("Rutas configuradas:")
print(f"  MIND train TSV  → {MIND_TRAIN_TSV}  ({'OK' if os.path.exists(MIND_TRAIN_TSV) else 'FALTA'})")
print(f"  MIND dev TSV    → {MIND_DEV_TSV}  ({'OK' if os.path.exists(MIND_DEV_TSV) else 'FALTA'})")
print(f"  CC-News raw     → {RAW_CC}")
print(f"  CC-News Parquet → {PROC_CC}")
print(f"  MIND Parquet    → {PROC_MIND}")

## 1. Instalación de dependencias

In [ ]:
# Ejecuta esta celda solo la primera vez
# !pip install pyspark datasets

## 2. Inicialización de PySpark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

spark = (
    SparkSession.builder
    .appName("Ingesta-Noticias")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"PySpark {spark.version} iniciado — modo local[*]")

## 3. CC-News — Descarga y guardado en Parquet

La descarga tarda ~15-30 min dependiendo de tu conexión. Se guarda directo en el disco externo.

In [ ]:
from datasets import load_dataset

print("Descargando CC-News desde HuggingFace...")
cc_hf = load_dataset(
    "vblagoje/cc_news",
    cache_dir=RAW_CC,
    split="train"
)
print(f"CC-News descargado: {len(cc_hf):,} artículos")
print("Columnas:", cc_hf.column_names)

In [ ]:
# Convertir a Spark DataFrame, quedándonos solo con las columnas útiles
cc_pd = cc_hf.to_pandas()[["title", "text", "description", "url", "date", "domain"]]

cc_df = spark.createDataFrame(cc_pd)

# Limpieza básica: eliminar filas sin texto
cc_df = (
    cc_df
    .filter(F.col("text").isNotNull() & (F.length(F.col("text")) > 100))
    .withColumn("source", F.lit("cc_news"))
    .withColumn("doc_id", F.monotonically_increasing_id())
)

print(f"Filas válidas: {cc_df.count():,}")
cc_df.printSchema()

In [ ]:
# Guardar en Parquet — particionado por dominio para facilitar consultas
(
    cc_df
    .repartition(8)  # 8 particiones para simular distribución en nodos
    .write
    .mode("overwrite")
    .parquet(PROC_CC)
)

print(f"CC-News guardado en Parquet: {PROC_CC}")

# Verificación
cc_verificado = spark.read.parquet(PROC_CC)
print(f"Verificación — filas leídas: {cc_verificado.count():,}")

## 4. MIND Large — Carga y guardado en Parquet

MIND usa formato TSV. El archivo principal es `news.tsv` (artículos con etiquetas).

**Estructura en el repositorio:**
```
datos/
├── MINDlarge_train/
│   ├── news.tsv          ← 101,527 artículos
│   └── behaviors.tsv
└── MINDlarge_dev/
    ├── news.tsv          ← 72,023 artículos
    └── behaviors.tsv
```

In [ ]:
import os

# Verificar que los archivos existen
for path in [MIND_TRAIN_TSV, MIND_DEV_TSV]:
    existe = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if existe else 0
    print(f"{'OK' if existe else 'FALTA':5s}  {path}  ({size_mb:.1f} MB)")

In [ ]:
MIND_SCHEMA = StructType([
    StructField("news_id",           StringType(), True),
    StructField("category",          StringType(), True),
    StructField("subcategory",       StringType(), True),
    StructField("title",             StringType(), True),
    StructField("abstract",          StringType(), True),
    StructField("url",               StringType(), True),
    StructField("title_entities",    StringType(), True),
    StructField("abstract_entities", StringType(), True),
])

def load_mind_news(tsv_path, split_name):
    return (
        spark.read
        .option("sep", "\t")
        .option("header", "false")
        .schema(MIND_SCHEMA)
        .csv(tsv_path)
        .withColumn("split",  F.lit(split_name))
        .withColumn("source", F.lit("mind_large"))
        .withColumn("text",   F.concat_ws(" ", F.col("title"), F.col("abstract")))
        .drop("title_entities", "abstract_entities")
        .filter(F.col("title").isNotNull() & F.col("category").isNotNull())
    )

mind_train = load_mind_news(MIND_TRAIN_TSV, "train")
mind_valid = load_mind_news(MIND_DEV_TSV,   "dev")

mind_df = mind_train.unionByName(mind_valid)

print(f"MIND train: {mind_train.count():,} artículos")
print(f"MIND dev:   {mind_valid.count():,} artículos")
print(f"Total MIND: {mind_df.count():,} artículos")
mind_df.printSchema()

In [ ]:
# Distribución de categorías
print("Distribución de categorías en MIND:")
mind_df.groupBy("category").count().orderBy(F.desc("count")).show(20, truncate=False)

In [ ]:
# Guardar en Parquet — particionado por category para acelerar Fase 4
(
    mind_df
    .repartition(4)
    .write
    .mode("overwrite")
    .partitionBy("category")
    .parquet(PROC_MIND)
)

print(f"MIND guardado en Parquet: {PROC_MIND}")

# Verificación
mind_verificado = spark.read.parquet(PROC_MIND)
print(f"Verificación — filas leídas: {mind_verificado.count():,}")

## 5. Resumen del almacenamiento

Análisis de tamaño y estructura — conecta con el tema de **Almacenamiento Distribuido** del curso.

In [ ]:
import subprocess

def get_dir_size(path):
    result = subprocess.run(["du", "-sh", path], capture_output=True, text=True)
    return result.stdout.split()[0] if result.returncode == 0 else "N/A"

print("=" * 55)
print("RESUMEN DE ALMACENAMIENTO")
print("=" * 55)
print(f"CC-News Parquet:  {get_dir_size(PROC_CC):>8s}  → {PROC_CC}")
print(f"MIND Parquet:     {get_dir_size(PROC_MIND):>8s}  → {PROC_MIND}")
print("=" * 55)

print("\nEstadísticas del corpus:")
print(f"  CC-News:  {cc_verificado.count():>8,} artículos  (sin etiquetas — para LSH)")
print(f"  MIND:     {mind_verificado.count():>8,} artículos  (18 categorías — para MLP)")

print("\nParticiones Parquet (simulación de bloques HDFS):")
cc_parts  = len(os.listdir(PROC_CC))
print(f"  CC-News:  {cc_parts} archivos Parquet")
print(f"  MIND:     particionado por categoría (1 carpeta por clase)")

## 6. Análisis costo-comunicación — Fase 1

Conexión con el modelo teórico del curso.

In [ ]:
cc_count   = cc_verificado.count()
mind_count = mind_verificado.count()
total      = cc_count + mind_count

print("MODELO COSTO-COMUNICACIÓN — INGESTA (Fase 1)")
print("-" * 50)
print(f"Total documentos ingestados : {total:,}")
print(f"Factor de replicación sim.  : 3 (estándar HDFS)")
print(f"Costo comunicación teórico  : O(n) — cada doc se lee 1 vez")
print(f"Costo almacenamiento        : O(n) sin replicación")
print()
print("Ventaja Parquet vs JSON plano:")
print("  - Lectura columnar: solo carga columnas necesarias → menos I/O")
print("  - Compresión Snappy: reduce transferencia de red en clúster")
print("  - Predicado pushdown: filtra antes de deserializar")

In [ ]:
spark.stop()
print("SparkSession cerrada. Fase 1 completada.")
print("Siguiente paso: 02_preprocesamiento.ipynb")